# Notebook 05: Golden Path
## Raw Agency Note → Canonical Packet → Decision → Session Strategy → Prompt Bundle

This notebook proves the full spine works end-to-end.

One real messy agency note flows through the entire pipeline:
- NB01: Extract facts, hypotheses, unknowns
- NB02: Detect blockers, contradictions, decide next action
- NB03: Plan the next interaction with session strategy and prompt bundle

Each step is explained in plain English.

## The Raw Input

This is a real agency note — the kind an agency owner jots down after a call or WhatsApp message.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.getcwd()))

from data.fixtures.raw_fixtures import RAW_FIXTURES

# Use the clean family booking fixture — a complete, happy-path scenario
fixture = RAW_FIXTURES["clean_family_booking"]

print("=" * 60)
print("  RAW AGENCY NOTE")
print("=" * 60)
print()
print(fixture["raw_input"])

## Step 1: Source Envelope

The raw input gets wrapped in a Source Envelope — preserving provenance. In the real system, this comes from NB01. For this demo, we construct the envelope manually.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import List, Dict, Any, Optional
import json

@dataclass
class EvidenceRef:
    ref_id: str
    envelope_id: str
    evidence_type: str
    excerpt: str
    field_path: Optional[str] = None
    confidence: float = 1.0

@dataclass
class Slot:
    value: Any = None
    confidence: float = 0.0
    authority_level: str = "unknown"
    extraction_mode: str = "unknown"
    evidence_refs: List[EvidenceRef] = field(default_factory=list)
    notes: Optional[str] = None

@dataclass
class UnknownField:
    field_name: str
    reason: str

@dataclass
class CanonicalPacket:
    packet_id: str
    created_at: str
    last_updated: str
    facts: Dict[str, Slot] = field(default_factory=dict)
    derived_signals: Dict[str, Slot] = field(default_factory=dict)
    hypotheses: Dict[str, Slot] = field(default_factory=dict)
    unknowns: List[UnknownField] = field(default_factory=list)
    contradictions: List[Dict[str, Any]] = field(default_factory=list)
    source_envelope_ids: List[str] = field(default_factory=list)
    stage: str = "discovery"

print("Source envelope created from raw note.")
print(f"  Source type: {fixture['source_type']}")
print(f"  Characters: {len(fixture['raw_input'])}")

## Step 2: Canonical Packet

NB01 extracts structured facts from the raw note. Each fact has:
- **value** — what was extracted
- **authority_level** — where it came from (explicit_owner, explicit_user, etc.)
- **confidence** — how sure the extraction is

In [ ]:
# Construct the packet from fixture expectations
expected = fixture["expected"]
extracted = expected["extracted_fields"]

facts = {}
for field_name, info in extracted.items():
    facts[field_name] = Slot(
        value=info["value"],
        confidence=0.9,
        authority_level=info["authority"],
    )

packet = CanonicalPacket(
    packet_id=fixture["fixture_id"],
    created_at=datetime.now().isoformat(),
    last_updated=datetime.now().isoformat(),
    facts=facts,
    unknowns=[],
    contradictions=[],
    stage="discovery",
)

print("=" * 60)
print("  CANONICAL PACKET")
print("=" * 60)
print()
print(f"Packet ID: {packet.packet_id}")
print(f"Stage: {packet.stage}")
print(f"Facts extracted: {len(packet.facts)}")
print()
for key, slot in packet.facts.items():
    print(f"  {key}: {slot.value}")
    print(f"    → authority: {slot.authority_level}, confidence: {slot.confidence}")
print()
print(f"Unknowns: {len(packet.unknowns)}")
print(f"Contradictions: {len(packet.contradictions)}")

## Step 3: Unknowns and Contradictions

What's missing? What conflicts?

In this clean case: nothing is missing, nothing conflicts. All 4 hard blockers are filled (origin, destination, dates, traveler count). All 3 soft blockers are also filled (budget, purpose, preferences).

In [ ]:
print("=" * 60)
print("  UNKNOWNS & CONTRADICTIONS")
print("=" * 60)
print()

if packet.unknowns:
    print("Unknowns (missing fields):")
    for u in packet.unknowns:
        print(f"  ✗ {u.field_name}: {u.reason}")
else:
    print("  ✓ No unknowns — all expected fields are present.")

print()

if packet.contradictions:
    print("Contradictions (conflicting values):")
    for c in packet.contradictions:
        print(f"  ✗ {c['field_name']}: {c['values']}")
else:
    print("  ✓ No contradictions — all values are consistent.")

## Step 4: NB02 Decision Result

The Decision Engine checks:
1. Any critical contradictions? → STOP
2. Any hard blockers missing? → ASK_FOLLOWUP
3. Any contradictions that suggest branching? → BRANCH_OPTIONS
4. All blockers filled + confidence high? → PROCEED_TRAVELER_SAFE
5. Soft blockers remaining? → PROCEED_INTERNAL_DRAFT

In [ ]:
# Load NB02 models
import sys, os
sys.path.insert(0, os.path.abspath(os.getcwd()))

def load_nb(path):
    with open(path, 'r') as f:
        nb = json.load(f)
    ns = {}
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            exec(cell['source'], ns)
    return ns

nb02 = load_nb('notebooks/02_gap_and_decision.ipynb')
run_gap_and_decision = nb02['run_gap_and_decision']

decision = run_gap_and_decision(packet)

print("=" * 60)
print("  NB02 DECISION RESULT")
print("=" * 60)
print()
print(f"Decision state: {decision.decision_state}")
print(f"Confidence: {decision.confidence_score:.0%}")
print(f"Hard blockers: {decision.hard_blockers if decision.hard_blockers else 'None (all filled)'}")
print(f"Soft blockers: {decision.soft_blockers if decision.soft_blockers else 'None (all filled)'}")
print(f"Contradictions: {len(decision.contradictions)}")
print(f"Follow-up questions: {len(decision.follow_up_questions)}")
print()
print(f"Rationale: {decision.rationale.get('reason', 'N/A')}")

## Step 5: NB03 Session Strategy

The Session Strategy answers: **What should the agent do next?**

Since the decision is PROCEED_TRAVELER_SAFE, the strategy is:
- Generate a complete proposal
- No hedging, no assumptions
- Present with confidence

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.getcwd()))

nb03 = load_nb('notebooks/03_session_strategy.ipynb')
build_session_output = nb03['build_session_output']
SessionOutput = nb03['SessionOutput']

output = build_session_output(decision, packet)

print("=" * 60)
print("  NB03 SESSION STRATEGY")
print("=" * 60)
print()
print(f"Session goal: {output.strategy.session_goal}")
print(f"Next action: {output.strategy.next_action}")
print()
print("Priority sequence:")
for i, step in enumerate(output.strategy.priority_sequence, 1):
    print(f"  {i}. {step}")
print()
print("Tonal guardrails:")
for g in output.strategy.tonal_guardrails:
    print(f"  • {g}")
print()
if output.strategy.risk_flags:
    print("Risk flags:")
    for r in output.strategy.risk_flags:
        print(f"  ⚠ {r}")
else:
    print("Risk flags: None")
print()
print(f"Suggested opening: {output.strategy.suggested_opening}")
print()
print("Exit criteria:")
for c in output.strategy.exit_criteria:
    print(f"  ✓ {c}")

## Step 6: Prompt Bundle

The Prompt Bundle is the actual text that goes into the next interaction. It has two parts:
1. **System context** — instructions for the LLM
2. **User message** — what the traveler or agent actually sees

In [ ]:
print("=" * 60)
print("  NB03 PROMPT BUNDLE")
print("=" * 60)
print()

print("--- SYSTEM CONTEXT ---")
print(output.prompts.system_context)
print()

print("--- USER MESSAGE ---")
print(output.prompts.user_message)
print()

print("--- CONSTRAINTS (what NOT to do) ---")
for c in output.prompts.constraints:
    print(f"  ✗ {c}")
print()

print("--- INTERNAL NOTES (agent only, not traveler) ---")
print(output.prompts.internal_notes)

## What is Internal vs What is Traveler-Facing

This is the critical boundary.

**Traveler-facing** (from user_message):
> "Create a complete proposal for the traveler.
> Known facts: origin_city: Bangalore; destination_city: Singapore; ..."

This is safe to share. No mention of unknowns, hypotheses, or contradictions.

**Internal-only** (from internal_notes):
> Active hypotheses: 0
> Soft blockers: None

The agent sees this too, but the traveler never does.

**System context** is NEVER shown to the traveler. It's instructions for the LLM:
> "Do not mention unknowns, hypotheses, or contradictions"
> "All claims must be grounded in facts"

In [ ]:
print("=" * 60)
print("  BOUNDARY CHECK")
print("=" * 60)
print()

um = output.prompts.user_message.lower()
sc = output.prompts.system_context.lower()
in_notes = output.prompts.internal_notes.lower()

# User message should NOT contain internal concepts
checks = [
    ("User message mentions 'unknown'", "unknown" not in um),
    ("User message mentions 'hypothesis'", "hypothesis" not in um),
    ("User message mentions 'contradiction'", "contradiction" not in um),
    ("User message mentions 'blocker'", "blocker" not in um),
    ("System context has constraints", len(output.prompts.constraints) > 0),
    ("Internal notes exist (agent-only)", len(output.prompts.internal_notes) > 0),
]

for label, passed in checks:
    status = "✓" if passed else "✗"
    print(f"  {status} {label}")

all_passed = all(p for _, p in checks)
print()
if all_passed:
    print("  All boundary checks passed. The spine is clean.")
else:
    print("  Some boundary checks failed — internal data may be leaking.")

## Summary: The Full Spine

```
Raw Input:
  "Family of 4 from Bangalore — 2 adults, 2 kids (ages 8 and 12).
   Want to go to Singapore, 5 nights, March 15-20.
   Budget around 3 lakhs total."

    ↓ NB01 (extraction)

Canonical Packet:
  7 facts extracted, 0 unknowns, 0 contradictions
  origin: Bangalore, destination: Singapore
  dates: March 15-20, travelers: 4
  budget: 3L, preferences: kid-friendly, vegetarian

    ↓ NB02 (decision)

Decision Result:
  PROCEED_TRAVELER_SAFE (confidence: 90%)
  0 hard blockers, 0 soft blockers, 0 contradictions

    ↓ NB03 (session planning)

Session Strategy:
  Goal: "Present a complete, traveler-ready proposal."
  Tone: Direct, confident, no hedging.
  Opening: "I've put together a complete proposal..."

Prompt Bundle:
  System context: "Generate traveler-ready proposal..."
  User message: "Create a complete proposal for the traveler..."
  Constraints: "Do not mention unknowns, hypotheses..."
  Internal notes: Agent-only summary
```

**The spine works. Raw messy note → structured state → decision → planned interaction.**